# Asaase Sense: OpenAQ Data Exploration

## Objective

Explore air-quality monitoring locations around Accra using OpenAQ API v3.

The purpose of this exercise is to understand the source data before designing our ingestion pipeline.

In [57]:
import os

import httpx
import pandas as pd

from dotenv import load_dotenv

Load API Credentials

In [58]:
load_dotenv()

OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY")

if not OPENAQ_API_KEY:
    raise ValueError("OPENAQ_API_KEY was not found.")

print("OpenAQ API key loaded successfully.")

OpenAQ API key loaded successfully.


Define API Configuration

In [59]:
BASE_URL = "https://api.openaq.org/v3"

HEADERS = {"X-API-Key": OPENAQ_API_KEY}

TIMEOUT = 30.0

Test API Connection

In [60]:
response = httpx.get(
    f"{BASE_URL}/countries", headers=HEADERS, params={"limit": 1}, timeout=TIMEOUT
)

response.raise_for_status()

print("Status code:", response.status_code)
print("Requested URL:", response.url)

Status code: 200
Requested URL: https://api.openaq.org/v3/countries?limit=1


Retrieve all countries

In [62]:
countries_response = httpx.get(
    f"{BASE_URL}/countries",
    headers=HEADERS,
    params={"limit": 1000, "page": 1},
    timeout=TIMEOUT,
)

countries_response.raise_for_status()

countries_data = countries_response.json()

country_records = countries_data["results"]

print("Total countries returned:", len(country_records))
print("Total countries reported by API:", countries_data["meta"]["found"])

Total countries returned: 159
Total countries reported by API: 159


Convert to dataframe

In [63]:
countries_df = pd.DataFrame(country_records)

countries_df = countries_df[["id", "code", "name"]]

display(countries_df)

,id,code,name
0,1,ID,Indonesia
1,2,MY,Malaysia
2,3,CL,Chile
3,4,BO,Bolivia
4,5,PE,Peru
...,...,...,...
154,238,TV,Tuvalu
155,239,MV,Maldives
156,243,FK,Falkland Islands / Malvinas
157,248,GU,Guam


Define African Country Codes

In [64]:
african_country_codes = {
    "DZ",
    "AO",
    "BJ",
    "BW",
    "BF",
    "BI",
    "CV",
    "CM",
    "CF",
    "TD",
    "KM",
    "CG",
    "CD",
    "CI",
    "DJ",
    "EG",
    "GQ",
    "ER",
    "SZ",
    "ET",
    "GA",
    "GM",
    "GH",
    "GN",
    "GW",
    "KE",
    "LS",
    "LR",
    "LY",
    "MG",
    "MW",
    "ML",
    "MR",
    "MU",
    "MA",
    "MZ",
    "NA",
    "NE",
    "NG",
    "RW",
    "ST",
    "SN",
    "SC",
    "SL",
    "SO",
    "ZA",
    "SS",
    "SD",
    "TZ",
    "TG",
    "TN",
    "UG",
    "ZM",
    "ZW",
}

print("African countries in our reference list:", len(african_country_codes))

African countries in our reference list: 54


Filter African Countries present in OpenAQ

In [ ]:
african_countries_df = countries_df[
    countries_df["code"].isin(african_country_codes)
].copy()

african_countries_df = (
    african_countries_df[["code", "name"]].sort_values("name").reset_index(drop=True)
)

display(african_countries_df)

print("African countries found in OpenAQ:", len(african_countries_df))

,code,name
0,DZ,Algeria
1,AO,Angola
2,BF,Burkina Faso
3,CM,Cameroon
4,TD,Chad
5,CI,Côte d'Ivoire
6,CD,Democratic Republic of the Congo
7,EG,Egypt
8,ET,Ethiopia
9,GH,Ghana


African countries found in OpenAQ: 35


Identify African countries missing from OpenAQ

In [66]:
available_codes = set(countries_df["code"])

missing_african_codes = african_country_codes - available_codes

print("African countries not currently listed by OpenAQ:")
print(sorted(missing_african_codes))

print("\nTotal not currently listed:", len(missing_african_codes))

African countries not currently listed by OpenAQ:
['BI', 'BJ', 'BW', 'CF', 'CG', 'DJ', 'ER', 'GA', 'GQ', 'GW', 'KM', 'LY', 'MR', 'NA', 'SC', 'SO', 'ST', 'SZ', 'TG']

Total not currently listed: 19


Retrieve locations for Ghana

In [67]:
ghana_response = httpx.get(
    f"{BASE_URL}/locations",
    headers=HEADERS,
    params={"iso": "GH", "limit": 1000, "page": 1},
    timeout=TIMEOUT,
)

ghana_response.raise_for_status()

ghana_data = ghana_response.json()

ghana_location_records = ghana_data["results"]

print("Country:", "Ghana")
print("Locations returned:", len(ghana_location_records))
print("Locations reported by API:", ghana_data["meta"]["found"])

Country: Ghana
Locations returned: 105
Locations reported by API: 105


In [68]:
location_columns = [
    "id",
    "name",
    "locality",
    "country.code",
    "country.name",
    "coordinates.latitude",
    "coordinates.longitude",
    "isMobile",
    "isMonitor",
    "datetimeFirst.utc",
    "datetimeLast.utc",
]

ghana_locations_df = ghana_locations_df[location_columns]

display(ghana_locations_df)

,id,name,locality,country.code,country.name,coordinates.latitude,coordinates.longitude,isMobile,isMonitor,datetimeFirst.utc,datetimeLast.utc
0,3,NMA - Nima,NaN,GH,Ghana,5.583890,-0.199680,False,True,NaN,NaN
1,4,NMT - Nima,NaN,GH,Ghana,5.581650,-0.198980,False,True,NaN,NaN
2,5,JTA - Jamestown,NaN,GH,Ghana,5.540114,-0.210397,False,True,NaN,NaN
3,6,ADT - Asylum Down,NaN,GH,Ghana,5.570722,-0.212056,False,True,NaN,NaN
4,7,ADEPA - Asylum Down,NaN,GH,Ghana,5.567833,-0.204028,False,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
100,6494903,Tema Community 3,NaN,GH,Ghana,5.620955,-0.023781,False,False,2026-08-18T14:00:00Z,2026-11-13T14:00:00Z
101,6521722,Tema Community 25 Estate 1,NaN,GH,Ghana,5.724653,0.017343,False,False,2026-09-03T15:00:00Z,2026-11-13T14:00:00Z
102,6542025,GAYO-119,NaN,GH,Ghana,5.651428,-0.185655,False,False,2026-09-15T19:03:58Z,2026-10-04T22:56:06Z
103,6542026,GAYO-120,NaN,GH,Ghana,5.651435,-0.185650,False,False,2026-09-15T19:02:29Z,2026-10-04T22:56:03Z


In [69]:
print("Total locations:", len(ghana_locations_df))
print("Unique locations:", ghana_locations_df["id"].nunique())
print("Mobile locations:", ghana_locations_df["isMobile"].sum())
print("Stationary locations:", (~ghana_locations_df["isMobile"]).sum())

Total locations: 105
Unique locations: 105
Mobile locations: 0
Stationary locations: 105


In [70]:
print("API-reported total:", ghana_data["meta"]["found"])
print("Page size:", ghana_data["meta"]["limit"])
print("Current page:", ghana_data["meta"]["page"])
print("Records retrieved:", len(ghana_location_records))

API-reported total: 105
Page size: 1000
Current page: 1
Records retrieved: 105


In [71]:
display(
    ghana_locations_df[
        ["name", "locality", "coordinates.latitude", "coordinates.longitude"]
    ].head(40)
)

,name,locality,coordinates.latitude,coordinates.longitude
0,NMA - Nima,NaN,5.583890,-0.199680
1,NMT - Nima,NaN,5.581650,-0.198980
2,JTA - Jamestown,NaN,5.540114,-0.210397
3,ADT - Asylum Down,NaN,5.570722,-0.212056
4,ADEPA - Asylum Down,NaN,5.567833,-0.204028
5,ADA - Asylum Down,NaN,5.566722,-0.207778
6,ELC - East Legon,NaN,5.633557,-0.165187
7,ELT - East Legon,NaN,5.640899,-0.169544
8,ELA - East Legon,NaN,5.640814,-0.157906
9,US Diplomatic Post: Accra,Accra,5.579447,-0.170699


In [72]:
print("API-reported locations:", ghana_data["meta"]["found"])
print("Records retrieved:", len(ghana_location_records))

API-reported locations: 105
Records retrieved: 105
